# Native 52M quick verification

Run this on Kaggle after the native 52M pretraining notebook. It needs the original YAML and `native_52m_pretrain.safetensors` checkpoint. This is intentionally a short GPU check: Triton RMSNorm/SwiGLU execution, then native-model SFT.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/AndyMLAndAI/picotron.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'transformers'], check=True)
os.chdir(REPO)

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
assert torch.cuda.get_device_capability(0) == (7, 5), 'This verification is sized for a T4/Turing GPU.'
DEVICE = torch.device('cuda:0')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


In [ ]:
from dataclasses import replace
from safetensors.torch import load_file
from transformers import AutoTokenizer

from picotron.config import load_config
from picotron.models.picotron_decoder import PicotronDecoderModel

# Override these only if the earlier pretraining notebook used different locations.
CONFIG_PATH = Path(os.environ.get('PICOTRON_52M_CONFIG', REPO / 'config_pretrain_52m.yaml'))
CHECKPOINT = Path(os.environ.get('PICOTRON_52M_CHECKPOINT', REPO / 'checkpoints/native_52m_pretrain.safetensors'))
assert CONFIG_PATH.exists(), f'Missing original 52M config: {CONFIG_PATH}'
assert CHECKPOINT.exists(), f'Missing native checkpoint: {CHECKPOINT}'

base_config = load_config(CONFIG_PATH)
# Keep architecture identical to the checkpoint; shorten only this verification run.
config = replace(
    base_config,
    model=replace(
        base_config.model,
        triton_kernels=replace(base_config.model.triton_kernels, rmsnorm=True, swiglu=True),
    ),
    tokens=replace(base_config.tokens, sequence_length=128, micro_batch_size=1, train_steps=25),
    logging=replace(base_config.logging, file_logging=False),
)
tokenizer = AutoTokenizer.from_pretrained(config.data.tokenizer_name or 'gpt2', use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_native_checkpoint():
    model = PicotronDecoderModel(config).to(DEVICE)
    model.load_state_dict(load_file(str(CHECKPOINT), device='cuda:0'))
    return model

@torch.no_grad()
def generate_native(model, prompt, max_new_tokens=48):
    # Native generation intentionally recomputes the full context: no KV cache yet.
    token_ids = tokenizer.encode(prompt, add_special_tokens=False)
    for _ in range(max_new_tokens):
        context = torch.tensor([token_ids[-config.tokens.sequence_length:]], device=DEVICE)
        next_token = model(context)[0, -1].argmax().item()
        token_ids.append(next_token)
        if next_token == tokenizer.eos_token_id:
            break
    return tokenizer.decode(token_ids)


In [ ]:
# 25 fp16-autocast steps exercise Triton forward + the real backward paths.
import warnings
from torch.nn import functional as F
from torch.optim import AdamW
from picotron.models.picotron_decoder import RMSNorm
from picotron.nn.feedforward import SwiGLU

model = load_native_checkpoint().train()
optimizer = AdamW(model.parameters(), lr=1e-5)
scaler = torch.amp.GradScaler('cuda')
losses = []
torch.manual_seed(1337)
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter('always')
    for step in range(25):
        input_ids = torch.randint(0, config.model.model_config.vocab_size, (1, 128), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids)
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), input_ids[:, 1:].reshape(-1))
        assert torch.isfinite(loss), f'non-finite loss at step {step}: {loss.item()}'
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())

fallback_messages = [str(item.message) for item in caught_warnings if 'Triton RMSNorm unavailable' in str(item.message) or 'Triton SwiGLU unavailable' in str(item.message)]
rmsnorm_requested = sum(isinstance(module, RMSNorm) and module.use_triton_rmsnorm for module in model.modules())
swiglu_requested = sum(isinstance(module, SwiGLU) and module.use_triton_swiglu for module in model.modules())
rmsnorm_fallbacks = sum(isinstance(module, RMSNorm) and module._triton_fallback_warned for module in model.modules())
swiglu_fallbacks = sum(isinstance(module, SwiGLU) and module._triton_fallback_warned for module in model.modules())
print(f'Triton check: requested RMSNorm={rmsnorm_requested}, SwiGLU={swiglu_requested}; fallback modules: RMSNorm={rmsnorm_fallbacks}, SwiGLU={swiglu_fallbacks}')
print(f'loss: first={losses[0]:.4f}, last={losses[-1]:.4f}, min={min(losses):.4f}, max={max(losses):.4f}')
assert rmsnorm_requested > 0 and swiglu_requested > 0
assert not fallback_messages and rmsnorm_fallbacks == 0 and swiglu_fallbacks == 0, (
    'FAIL: a Triton kernel fell back to PyTorch; inspect the warning above before claiming kernel execution.', fallback_messages
)
print('PASS: no Triton RMSNorm/SwiGLU fallback was observed during 25 forward/backward steps.')
del model, optimizer
torch.cuda.empty_cache()


In [ ]:
# Native Picotron checkpoint -> native SFT (256 SmolTalk examples, 30 steps).
import itertools
from datasets import load_dataset
from torch.utils.data import Dataset
from picotron_sft import run_sft

PROMPTS = ['Explain why the sky is blue.', 'Write Python that returns the nth Fibonacci number.']
sft_model = load_native_checkpoint().eval()
before = [generate_native(sft_model, prompt) for prompt in PROMPTS]

examples = list(itertools.islice(load_dataset('HuggingFaceTB/smoltalk', 'all', split='train', streaming=True), 256))
assert len(examples) == 256

class NativeSmolTalkDataset(Dataset):
    def __init__(self, source_examples):
        self.source_examples = source_examples
    def __len__(self):
        return len(self.source_examples)
    def __getitem__(self, index):
        messages = self.source_examples[index]['messages']
        text = '\n'.join(f"{message['role']}: {message['content']}" for message in messages)
        ids = tokenizer.encode(text, add_special_tokens=False)[:config.tokens.sequence_length - 1] + [tokenizer.eos_token_id]
        padding = config.tokens.sequence_length - len(ids)
        return (
            torch.tensor(ids + [tokenizer.pad_token_id] * padding, dtype=torch.long),
            torch.tensor(ids + [-100] * padding, dtype=torch.long),
        )

sft_model.train()
sft_losses = run_sft(
    sft_model, NativeSmolTalkDataset(examples), learning_rate=2e-5, weight_decay=0.0,
    batch_size=1, num_steps=30, device=DEVICE, display_config=config,
)
assert len(sft_losses) == 30 and all(torch.isfinite(torch.tensor(sft_losses)))
print(f'Native SFT PASS: first_loss={sft_losses[0]:.4f}, last_loss={sft_losses[-1]:.4f}')
sft_model.eval()
for prompt, pre_sft, post_sft in zip(PROMPTS, before, [generate_native(sft_model, item) for item in PROMPTS]):
    print(f'\nPROMPT: {prompt}\n[before SFT] {pre_sft}\n[after SFT]  {post_sft}')
print('PASS: native checkpoint completed direct Picotron SFT. Generation uses full-forward decoding (no KV cache).')
